## Setup Environment

First, we need to install the required libraries for fine-tuning the TinyLlama model.

In [ ]:
import torch
import os

# Install libraries. suppress output for cleaner logs
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

# Check if CUDA is available for GPU training
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

# Mount Google Drive if you plan to store your dataset or model there
# from google.colab import drive
# drive.mount('/content/drive')

# If you uploaded your file directly to Colab's session storage, it will be in the /content/ directory.
# Example if your file is named 'my_training_data.jsonl':
# DATASET_PATH = "my_training_data.jsonl"
# You can verify its presence using:
# !ls -lh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.3 MB/s eta 0:00:00
CUDA available: True
CUDA device name: Tesla T4


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# 1. Configuration
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_PATH = "all_combined.jsonl"  # <--- path to your JSONL file
OUTPUT_DIR = "./tinyllama_fine_tuned"

# QLoRA configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training arguments
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 3
MAX_SEQ_LENGTH = 512  # Adjust based on your data and GPU memory

# --- THE FIX: keep precision settings consistent ---
# T4 (free Colab) does NOT support bf16; A100/L4 do. Auto-detect so the
# compute dtype and the trainer's mixed-precision mode always agree.
USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"bf16 supported: {USE_BF16} -> compute dtype: {COMPUTE_DTYPE}")

# 2. Load the dataset
# Ensure your JSONL has a 'messages' field (list of {'role','content'} dicts).
try:
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    print(f"Dataset loaded successfully with {len(dataset)} examples.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure your JSONL file exists and is correctly formatted.")
    raise

# 3. Load the model and tokenizer

# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # was hard-coded bfloat16
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Set pad_token to eos_token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Set the model_max_length for the tokenizer to ensure proper truncation
tokenizer.model_max_length = MAX_SEQ_LENGTH

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},  # Use GPU 0 for training
)

# 4. Prepare model for QLoRA training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# 5. Define a formatting function for SFTTrainer
def formatting_func(examples):
    output_texts = []
    for i in range(len(examples["messages"])):  # iterate over each example in the batch
        output_texts.append(tokenizer.apply_chat_template(
            examples["messages"][i], tokenize=False, add_generation_prompt=False
        ))
    return output_texts

# 6. Set up training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not USE_BF16,   # fp16 only when bf16 is unavailable
    bf16=USE_BF16,       # bf16 when the GPU supports it
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none",
)

# 7. Initialize and start the Trainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    formatting_func=formatting_func,
)

print("Starting training...")
trainer.train()
print("Training complete!")

# 8. Save the fine-tuned model (LoRA adapters only)
trainer.save_model(OUTPUT_DIR)
print(f"Fine-tuned LoRA adapters saved to {OUTPUT_DIR}")

print("Fine-tuning script finished.")

bf16 supported: True -> compute dtype: torch.bfloat16
Dataset loaded successfully with 3657 examples.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


Applying formatting function to train dataset:   0%|          | 0/3657 [00:00<?, ? examples/s]

...

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,1.832563
100,0.681651
150,0.293660
200,0.179003
250,0.164881
300,0.152037
350,0.143848
400,0.138234
450,0.137993
500,0.133406


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training complete!
Fine-tuned LoRA adapters saved to ./tinyllama_fine_tuned
Fine-tuning script finished.


In [ ]:
!pip uninstall -y torchao
!pip install -U peft transformers accelerate bitsandbytes

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "./tinyllama_fine_tuned"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

messages = [
    {
        "role": "system",
        "content": """ You are an assistant that extracts and normalizes a user's birth date from input text.

        STRICT_RULE:
          - Return only one value in format: MM-YYYY
          - Month must always be 2 digits (01–12)
          - If date is unclear return RETRY

        IMPORTANT:
          - Handle different date formats (DD/MM/YYYY, MM/YYYY, Month YYYY, etc.)
          - Handle common typos and spacing issues
          - Convert textual months to numeric format
          - Ignore extra words like "DOB", "born", etc.

        MAPPING_RULES:
          - Extract month and year from input → convert to MM-YYYY
          - If only year is present → RETRY
          - If multiple interpretations possible → RETRY

        Examples:
          - 12/05/1995 -> 05-1995
          - 05/1995 -> 05-1995
          - May 1995 -> 05-1995
          - may-1995 -> 05-1995
          - 1995 May -> 05-1995
          - DOB 5 1995 -> 05-1995
          - 5-95 -> 05-1995
          - septo 88 -> 07-1988
          - janoary eity two-> 01-82

          - 1995 -> RETRY
          - I am 30 years old -> RETRY"""
    },
    {
        "role": "user",
        "content": "septo 89"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.1,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

result = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


01-1989
